# DLDR-Net for 16-QAM Detection: Annotated Implementation

This notebook implements a model-driven neural detector related to the paper *Intelligent Dynamic Relaxation Detection for RIS-Assisted Massive MIMO Systems* (DOI: 10.1109/TVT.2026.3756806). It converts a complex MIMO detection problem into real form, unfolds a momentum-based update into neural layers, trains their parameters, and compares bit error rate (BER) with a linear MMSE-style baseline.

**Reading order:** imports → data and real conversion → MMSE system → unfolded layers → loss and training → hard detection and BER → baseline and plot. The explanations refer to the paper's equations (1), (2), and (12)–(18).

**Execution prerequisites:** the notebook calls `Load_data()` but does not define it or include the dataset. Supply the original loader and data before running the data cell. The available TensorFlow/Keras environment must support the APIs used here. No RIS channel is generated explicitly in this notebook: the loader must supply the effective channel if an RIS-assisted experiment is intended.

**Scope:** these notes explain the existing code, including assumptions and limitations. The existing computational code and saved outputs have not been changed.

## 1. Libraries
NumPy handles numerical arrays, TensorFlow/Keras builds and trains the detector, Matplotlib draws figures, and pickle/random support serialization and randomness. Several imports are repeated; that does not change the algorithm. No random seed is set here, so repeated results need not be identical.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
import pickle as pkl

import numpy as np
import tensorflow as tf
# import tensorflow_probability as tfp
import matplotlib.pyplot as plt
from tensorflow import keras
import pickle
import random
# tf.debugging.set_log_device_placement(True)

## 2. System configuration and data

- `Nt = 8`: number of transmitted complex streams/users; `Nr = 32`: number of complex receive antennas.
- `QAM = 16`: each complex symbol carries four bits. Each real or imaginary component has four amplitude levels.
- `SNR_DB = np.arange(10,22,2)` gives 10, 12, 14, 16, 18, and 20 dB; the endpoint 22 is excluded.
- `Num_cc`, `Num_ii`, `cesit`, `cesir`, and `channel_error_var` are set here but are not subsequently used in the visible code. Their effect, if any, depends on the missing loader or external context.
- `Load_data()` should return the transmitted symbols `s`, effective complex channel `H`, received signal `r`, and noise `n`, each accessible by SNR key. The actual dataset format and whether it includes the RIS cannot be verified from this notebook alone.

For the unnormalized 16-QAM alphabet, the average complex-symbol energy is $E=(2/3)(M-1)=10$. The later activation and slicer use $\sqrt E$ to convert between normalized amplitudes and the raw levels $\{-3,-1,1,3\}$. This interpretation assumes the loaded symbols were normalized consistently.

In [ ]:
Num_cc = np.arange(0,10)
Num_ii = 100
SNR_DB = np.arange(10,22,2)
Nt = 8
Nr = 32
QAM = 16
cesit = .0
cesir = .0
channel_error_var=0.0

s, H, r, n = Load_data()


E = (QAM-1)*((2*2)/2)/3
E

## 3. Convert the complex system to real-valued form

The paper models the effective channel as $\widetilde H=\widetilde H_{BU}+\widetilde H_{BR}\Theta\widetilde H_{RU}$ and the received signal as $\widetilde y=\widetilde H\widetilde s+\widetilde n$. With $\widetilde H=H_R+jH_I$, the equivalent real system is

$$\begin{bmatrix}\Re\widetilde y\\\Im\widetilde y\end{bmatrix}=\begin{bmatrix}H_R&-H_I\\H_I&H_R\end{bmatrix}\begin{bmatrix}\Re\widetilde s\\\Im\widetilde s\end{bmatrix}+\begin{bmatrix}\Re\widetilde n\\\Im\widetilde n\end{bmatrix}.$$

This is the paper's equation (2). `s_real`, `r_real`, `h_real`, and `n_real` implement this conversion, and the results for all SNRs are concatenated in SNR order. With batch size $B$, the intended shapes are `s_`: $(B,2N_t,1)$, `r_`: $(B,2N_r,1)$, and `H_`: $(B,2N_r,2N_t)$; the actual loader must supply compatible trailing dimensions. For $N_t=8$ and $N_r=32$, the real channel is $64\times16$.

In [ ]:
for snr in SNR_DB:
    sreal = np.real(s[snr])
    simag = np.imag(s[snr])
    s_real = np.concatenate((sreal,simag),1)

    rreal = np.real(r[snr])
    rimag = np.imag(r[snr])
    r_real = np.concatenate((rreal,rimag),1)

    hreal = np.real(H[snr])
    himag = np.imag(H[snr])
    hup = np.concatenate((hreal,-himag),2)
    hdown = np.concatenate((himag,hreal),2)
    h_real = np.concatenate((hup,hdown),1)

    nreal = np.real(n[snr])
    nimag = np.imag(n[snr])
    n_real = np.concatenate((nreal,nimag),1)

    if snr == SNR_DB[0]:

        s_ = s_real
        H_ = h_real
        r_ = r_real
        noise = n_real

    else:

        s_ = np.concatenate((s_ , s_real),0)
        H_ = np.concatenate((H_ , h_real),0)
        r_ = np.concatenate((r_ , r_real),0)
        noise = np.concatenate((noise , n_real),0)



‍## add effect of E

## 4. Build the linear detection system

The paper uses the regularized system

$$A=H^TH+\sigma^2I,\qquad d=H^Ty,\qquad As\approx d.$$

Here `A` represents $A$ and `b` represents the paper's $d$. `sigmaI` is a diagonal matrix whose coefficient is estimated by `np.var(noise, axis=1)` separately for each realization. Therefore, this notebook uses the sample variance of the real-form noise, not an independently specified noise variance. Its equivalence to the paper's regularization depends on the loader's normalization and noise convention, including real-versus-complex variance and symbol energy. The placeholder comment about the effect of `E` above remains unresolved by the code.

If $B$ is the number of samples, `A` has shape $(B,2N_t,2N_t)$ and `b` has shape $(B,2N_t,1)$.

In [ ]:
sigmaI = (np.var(noise,1,keepdims=True)*np.eye(s_.shape[1]))

A = (np.transpose(H_,(0,2,1)) @ H_+sigmaI)
b = np.transpose(H_,(0,2,1)) @ r_

## 5. Diagonal preconditioner

`np.tril(A)` and `np.triu(A)` both include the diagonal, so `D = -(A-U-L)` equals $\operatorname{diag}(A)$. Thus `M_inv = inv(D)` is the inverse of the diagonal of $A$. In the paper's notation, it plays the role of $X^{-1}$, where $X=\operatorname{diag}(A)$.

The preconditioner scales each coordinate of the residual by its diagonal coefficient. Because $D$ is diagonal, a reciprocal of its diagonal entries could avoid a general matrix inverse, but the existing implementation uses `np.linalg.inv`. Inversion requires nonzero diagonal entries.

In [ ]:
L = np.tril(A)
U = np.triu(A)
D = -(A-U-L)
M_inv = np.linalg.inv(D)

## 6. Split training and test samples

Even-indexed samples are assigned to training and odd-indexed samples to testing. Because the concatenation above is performed in SNR order, this is an alternating split, not a shuffled random split. The paper describes a 50/50 data split; the code does not establish whether samples from the same underlying channel realization or coherence block occur in both sets. If they do, test performance may overstate generalization to unseen channels.

The split preserves corresponding symbols, channels, matrices, and right-hand sides. The numbers of training and test samples can differ by one if the dataset length is odd.

In [ ]:
tr = np.arange(0,s_.shape[0],2)
te = np.arange(1,s_.shape[0],2)

s_tr = s_[tr]
s_te = s_[te]

H_tr = H_[tr]
H_te = H_[te]

A_tr = A[tr]
A_te = A[te]

b_tr = b[tr]
b_te = b[te]

M_i_tr = M_inv[tr]
M_i_te = M_inv[te]

## 7. One unfolded dynamic-relaxation layer

Each `Custom_layer` implements the layer update in equations (13) and (14) of the paper:

$$v_k=w_{1,k}\odot v_{k-1}+w_{2,k}\odot X^{-1}(d-As_{k-1}),$$
$$s_k=\Psi(s_{k-1}+v_k+\beta_k).$$

Code-to-equation mapping:

| Notebook | Meaning |
|---|---|
| `x` | previous symbol estimate $s_{k-1}$ |
| `v` | previous velocity/momentum $v_{k-1}$ |
| `A`, `b` | system matrix $A$ and right-hand side $d$ |
| `M_inv` | diagonal preconditioner $X^{-1}$ |
| `W1` | learned per-coordinate momentum vector $w_{1,k}$ |
| `W2` | learned per-coordinate correction vector $w_{2,k}$ |
| `l` | learned bias vector $\beta_k$ |

Each vector has shape $(2N_t,1)$ and is broadcast across samples. `W1` and `W2` are distinct trainable vectors in every layer; the bias is a third trainable vector. The paper emphasizes two weight vectors while also specifying a bias. With $K$ layers, this implementation has $3K(2N_t)$ trainable scalar entries, excluding any external model parameters.

`Model_train` applies layers sequentially, saves all estimates and velocities, and returns `(all_layer_estimates, final_estimate, all_velocities)`. For a batch of size $B$, their intended shapes are $(B,2N_t,K)$, $(B,2N_t,1)$, and $(B,2N_t,K)$.

In [ ]:
class Custom_layer(keras.layers.Layer):
    def __init__(self , act):
        super(Custom_layer, self).__init__()

        self.act = act
        self.W1 = self.add_weight(initializer="normal", shape=(2*Nt,1),dtype="float32", trainable=True, name = 'W1_')#此為tf的建立方法
        self.W2 = self.add_weight(initializer="normal", shape=(2*Nt,1),dtype="float32", trainable=True, name = 'W2_')#此為tf的建立方法
        self.l  = self.add_weight(initializer="normal", shape=(2*Nt,1),dtype="float32", trainable=True, name = 'l_')#此為tf的建立方法


    def call(self, inputs):
        x, A, b, M_inv, v = inputs[0], inputs[1], inputs[2], inputs[3],inputs[4]

        v = self.W1*v+self.W2*(M_inv@(b-A@x))

        X = x +v

        return self.act(X+self.l),v


class Model_train(keras.Model):
    def __init__(self, model):
        super(Model_train, self).__init__()

        self.model = model

    def call(self, inputs):
        x, A, b, M_inv, v = inputs[0], inputs[1], inputs[2], inputs[3],inputs[4]
        List = []
        V=[]
        for i in range(len(self.model)):
            x, v = self.model[i]([x, A, b, M_inv, v])
            List.append(x)
            V.append(v)

        S = tf.concat(List,2)

        return S,x,tf.concat(V,2)


## 8. Network construction, activation, and training loss

`built_net` creates independent layer objects and repeats the ground-truth symbol vector along the layer axis so every layer is supervised.

For 16-QAM, the paper uses a sum of three shifted hyperbolic tangents. The notebook applies the normalized form

$$\Psi(x)=\frac{\tanh(\varsigma\sqrt E x)+\tanh(\varsigma\sqrt E x-2\varsigma)+\tanh(\varsigma\sqrt E x+2\varsigma)}{\sqrt E},\quad\varsigma=1.5.$$

The smooth staircase encourages outputs near the four normalized real amplitudes while allowing gradient-based learning. `E` must agree with the normalization of the input labels.

`lf` calculates the squared Euclidean error per sample and layer, weights it by $\log(k+1)$ for $k=1,\ldots,K$, and **sums** over all samples and layers. This corresponds to the layer-weighting idea in equation (18). Because the code uses a sum rather than a sample mean, the numerical loss and gradient scale depend on the dataset size. The returned velocity tensor is assigned to `V` but is not used in this loss.

In [ ]:
def built_net(layer,s,Nt,act):
    model_list =[]
    S_list = []
    for i in range(layer):
        S_list.append(s)
        model_list.append(Custom_layer(act))

    return np.concatenate(S_list,2), model_list


def activation(x):
    x=np.sqrt(E)*x
    sig =1.5
    return (tf.tanh(sig*x)+tf.tanh(sig*x-2*sig)+tf.tanh(sig*x+2*sig))/np.sqrt(E)


def lf(outdv, Sd):
    outd,V=outdv[0],outdv[2]
    layer_ = Sd.shape[2]
    Sd = tf.constant(Sd,dtype=tf.float32)

    

    lf =   tf.norm(outd-Sd,ord=2,axis=1)**2


    return tf.math.reduce_sum(lf*np.log(np.arange(2,layer_+2)))


## 9. Quick diagnostic cells

The next cells inspect the number of SNR points, generate an illustrative random number, and plot the activation. They do not train the model. Since no seed is set, the random value varies between runs.

In [ ]:

len(SNR_DB)

In [ ]:
np.random.randn()

### Plot the 16-QAM activation

The horizontal axis is an input coordinate and the vertical axis is its soft symbol estimate. The steps approach the normalized amplitude levels. This is a soft differentiable mapping; the later nearest-level slicer produces hard bit decisions.

In [ ]:
x= np.linspace(-2,2,100)
plt.plot(x,activation(x))
activation(.5);

## 10. Number of unfolded layers

`layer = 15` means the iterative update is unfolded into 15 separately parameterized stages. More layers can improve iterative refinement but increase both computation and parameter count; performance must be measured rather than assumed. At $N_t=8$, this code creates $3\times15\times16=720$ trainable scalar entries once built.

In [ ]:
layer = 15

## 11. Build targets and layer objects

`S` and `S_te` repeat each reference vector across all 15 layers for layer-wise loss computation. `model_list` contains the actual 15 layer objects used to build the model. The second `built_net` call also constructs an unused list of new test layer objects; those objects are discarded and are not the trained model.

In [ ]:
S, model_list = built_net(layer,s_tr,Nt,activation)
S_te,_ = built_net(layer,s_te,Nt,activation)

## 12. Assemble the model

`Model_train(model_list)` wraps the sequence of independently parameterized layers into one Keras model. The same trained layers will be used at inference. The input contains an initial estimate, the per-sample system matrix and right-hand side, the diagonal preconditioner, and an initial velocity.

In [ ]:
Model = Model_train(model_list)

## 13. Evaluate the initial loss

The initial estimate and velocity are set to zero, as in the paper's forward algorithm. The call performs a forward pass and reports the layer-weighted training loss before the subsequent optimization. It also builds the model's trainable weights on first call in typical Keras execution.

In [ ]:
lf(Model([s_tr*0, A_tr, b_tr, M_i_tr, s_tr*0]),S).numpy()

## 14. Initial training and test loss

The cell evaluates both losses before training using the same network. **Shape caution:** `v` is created from `s_tr` and then reused with test data. This works only when the training and test batch shapes are compatible; otherwise use a test velocity based on `s_te * 0`. The same issue appears in the later test-inference cell. Summed training and test losses are directly comparable only when batch sizes and data weighting are comparable.

In [ ]:
v= s_tr*0
lf(Model([s_tr*0, A_tr, b_tr, M_i_tr, v]),S).numpy(),lf(Model([s_te*0, A_te, b_te, M_i_te, v]),S_te).numpy()

## 15. Adam optimizer

`lr = 0.001` is the Adam learning rate, and the initial training velocity is zero. Adam updates the learned layer vectors using gradients of the loss. The training loop below uses `tf.GradientTape` and `apply_gradients`, which avoids the unavailable `optimizer.minimize` method observed in some Keras versions. This code performs full-batch updates; it does not implement minibatch sampling.

In [ ]:
lr = 0.001
opt = tf.keras.optimizers.Adam(learning_rate=lr)

v = s_tr * 0

## 16. Train the network

For each of 5,000 optimization steps:

1. Run the 15-layer detector and compute the supervised loss inside `GradientTape`.
2. Differentiate the loss with respect to `Model.trainable_weights`.
3. Apply gradients with Adam.
4. Print the loss computed **before** that step's weight update.

This loop prints 5,000 lines and does not track validation loss, save a checkpoint, or stop early. Although it runs 5,000 full-batch steps, that is not automatically equivalent to the paper's epoch/minibatch setup. Training requires all operations in the forward pass and loss to remain differentiable TensorFlow operations.

In [ ]:


for i in range(5000):

    # Calculate the loss and record operations for differentiation
    with tf.GradientTape() as tape:
        predictions = Model([s_tr * 0, A_tr, b_tr, M_i_tr, v])
        loss = lf(predictions, S)

    # Calculate and apply gradients
    gradients = tape.gradient(loss, Model.trainable_weights)
    opt.apply_gradients(zip(gradients, Model.trainable_weights))

    print("ep:", i, "Loss:", loss.numpy())

## 17. Detect test symbols

The model's second return value (`[1]`) is the final-layer soft estimate. The code uses the trained model with test matrices and observations. As noted above, the velocity is initialized from `s_tr`, not `s_te`; it must have a shape compatible with the test batch. The model output is still continuous, so hard slicing is performed next.

In [ ]:
v= s_tr*0

s_out = Model([s_te*0, A_te, b_te, M_i_te, v])[1]


## 18. Hard slicing and bit mapping

Multiplying the normalized output by $\sqrt E$ maps it back to the raw real-axis levels $[-3,-1,1,3]$. For every real and imaginary coordinate, the code chooses the nearest level and maps it to the corresponding two-bit label in `sym`.

The labels used here are `00, 01, 10, 11`. This is a particular binary mapping, **not Gray coding** because the adjacent middle levels differ in two bits. BER depends on the labeling; reproducing a paper or dataset requires the mapping to match the transmitter's bit-to-symbol mapping. Here reference symbols are mapped using the same table, but that does not establish the original transmitter's bit mapping.

In [ ]:
M = np.array([-3,-1,1,3])
sym = np.array([[0,0],[0,1],[1,0],[1,1]])
S_Out = sym[np.expand_dims(np.argmin(np.abs(s_out*np.sqrt(E)-M),2),2)]
S_True = sym[np.expand_dims(np.argmin(np.abs(s_te*np.sqrt(E)-M),2),2)]

## 19. Compute test BER by SNR

`er` is a Boolean array indicating bit disagreements. A complex 16-QAM symbol carries 4 bits, so one example contains $4N_t$ bits; the denominator `l * (4*Nt)` counts the bits for `l` examples per SNR. The code splits the test array into equal contiguous blocks, assuming the same number of test samples at every SNR and preserved SNR order. If the loader supplies unequal counts or the alternating split crosses block boundaries unevenly, these BER assignments can be incorrect. A robust experiment should retain explicit SNR labels for each sample.

In [ ]:
er = S_Out != S_True
l = int(er.shape[0]/len(SNR_DB))

ber = []
for i in range(len(SNR_DB)):
    be = np.sum(er[i*l:(i+1)*l])/l/(4*Nt)

    ber.append(be)



### Inspect numerical BER values

`ber` contains one estimated test BER per SNR. A zero value means that no errors were observed in the finite test set; it does not prove that the true BER is zero. Reliable low-BER estimation requires enough transmitted bits and observed errors.

In [ ]:
ber


## 20. Linear MMSE-style baseline

The baseline solves $As=d$ using `np.linalg.inv(A_mmse) @ b`, then applies the same nearest-level slicer and bit labels. This makes a useful reference for the unfolded network. Numerical practice generally favors `np.linalg.solve(A_mmse, b)` over explicitly forming an inverse, though the original code is retained.

**Comparison caveat:** the neural BER above is computed on the test subset, while `ber_m` here uses **all samples**, including training data. The per-SNR block calculation also assumes equal sample counts. For a fair comparison, evaluate both methods on the same held-out samples and use their explicit SNR labels. The label-mapping and variance-normalization caveats apply to both.

In [ ]:
A_mmse = (np.transpose(H_,(0,2,1)) @ H_+sigmaI)
s_mmse = np.linalg.inv(A_mmse)@b

M = np.array([-3,-1,1,3])
sym = np.array([[0,0],[0,1],[1,0],[1,1]])
S_m = sym[np.expand_dims(np.argmin(np.abs(s_mmse*np.sqrt(E)-M),2),2)]
S_TTrue = sym[np.expand_dims(np.argmin(np.abs(s_*np.sqrt(E)-M),2),2)]

er = S_m != S_TTrue
l = int(er.shape[0]/len(SNR_DB))

ber_m = []
for i in range(len(SNR_DB)):
    be = np.sum(er[i*l:(i+1)*l])/l/(4*Nt)

    ber_m.append(be)

print(ber_m)
print('Hi')

## 21. Plot BER against SNR

A semilogarithmic plot places BER on a logarithmic vertical axis and SNR in dB on the horizontal axis. The plotted curves compare the final unfolded network and the linear baseline under this notebook's data processing. Zero empirical BER values cannot be displayed on a log axis. Interpret the comparison only after addressing the dataset, bit mapping, per-SNR grouping, and differing test/all-data evaluation noted above.

**What this notebook does not implement visibly:** explicit generation or optimization of RIS phases; the separate MDR procedure based on eigenvalue bounds; or additional paper baselines. Those components may exist externally, but they cannot be inferred from this file.

In [ ]:
plt.semilogy(SNR_DB,ber_m,'^-',label='MMSE')
plt.semilogy(SNR_DB,ber,'^-',label='DR_Net')
plt.title(str(layer)+' Layer '+str(Nr)+' * '+str(Nt)+' 16QAM')
plt.grid()
plt.legend()